# SILVER → GOLD: Agregacion de Customers

Este notebook ejecuta el script de agregacion que:
1. Lee la capa Silver (Parquet)
2. Agrupa por estado y calcula metricas de negocio
3. Guarda el resumen en la capa Gold (Parquet)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\Usuario\Documents\Cursos\programacion_con_ia\multihope_pa


## 1. Verificar que la capa Silver existe

In [3]:
SILVER_PATH = PROJECT_ROOT / 'data' / 'silver' / 'customers'

if not SILVER_PATH.exists():
    raise FileNotFoundError(
        f'Silver layer no encontrada en {SILVER_PATH}.\n'
        'Ejecuta primero el notebook 02_bronze_to_silver_customers.ipynb'
    )
print(f'Silver layer encontrada en: {SILVER_PATH}')

Silver layer encontrada en: C:\Users\Usuario\Documents\Cursos\programacion_con_ia\multihope_pa\data\silver\customers


## 2. Ejecutar el script SILVER → GOLD

In [3]:
from src.silver_to_gold.customers_aggregation import aggregate_customers

total_rows = aggregate_customers()
print(f'\nAgregacion completada: {total_rows} filas en Gold.')

2026-04-05 18:53:58,237 [INFO] src.silver_to_gold.customers_aggregation - === SILVER -> GOLD | customers_summary ===
2026-04-05 18:54:10,189 [INFO] src.silver_to_gold.customers_aggregation - Reading Silver layer from C:\Users\Usuario\Documents\Cursos\programacion_con_ia\multihope_pa\data\silver\customers
2026-04-05 18:54:18,346 [INFO] src.silver_to_gold.customers_aggregation - Building Gold aggregates...
2026-04-05 18:54:23,907 [INFO] src.silver_to_gold.customers_aggregation - Gold aggregation rows: 2
2026-04-05 18:54:23,908 [INFO] src.silver_to_gold.customers_aggregation - Saving to C:\Users\Usuario\Documents\Cursos\programacion_con_ia\multihope_pa\data\gold\customers_summary
2026-04-05 18:54:27,260 [INFO] src.silver_to_gold.customers_aggregation - Gold layer written successfully — 2 rows.



Agregacion completada: 2 filas en Gold.


## 3. Explorar la capa Gold

In [4]:
from src.utils.spark_session import create_spark_session

GOLD_PATH = PROJECT_ROOT / 'data' / 'gold' / 'customers_summary'

spark = create_spark_session('notebook_gold_validation')
df_gold = spark.read.parquet(str(GOLD_PATH))

print('Schema Gold:')
df_gold.printSchema()

Schema Gold:
root
 |-- estado: string (nullable = true)
 |-- total_customers: long (nullable = true)
 |-- unique_emails: long (nullable = true)
 |-- first_loadtime: timestamp (nullable = true)
 |-- last_loadtime: timestamp (nullable = true)
 |-- _created_at: timestamp (nullable = true)



In [5]:
print('Resumen de customers por estado:')
df_gold.orderBy('total_customers', ascending=False).show(truncate=False)

Resumen de customers por estado:
+--------+---------------+-------------+-------------------+-------------------+-----------------------+
|estado  |total_customers|unique_emails|first_loadtime     |last_loadtime      |_created_at            |
+--------+---------------+-------------+-------------------+-------------------+-----------------------+
|activo  |8              |8            |2025-06-26 19:34:48|2025-06-26 19:34:48|2026-04-05 18:54:23.977|
|inactivo|2              |2            |2025-06-26 19:34:48|2025-06-26 19:34:48|2026-04-05 18:54:23.977|
+--------+---------------+-------------+-------------------+-------------------+-----------------------+



In [6]:
# Visualizacion con pandas
df_pandas = df_gold.toPandas()
df_pandas.sort_values('total_customers', ascending=False)

,estado,total_customers,unique_emails,first_loadtime,last_loadtime,_created_at
0,activo,8,8,2025-06-26 19:34:48,2025-06-26 19:34:48,2026-04-05 18:54:23.977
1,inactivo,2,2,2025-06-26 19:34:48,2025-06-26 19:34:48,2026-04-05 18:54:23.977


In [7]:
spark.stop()
print('SparkSession cerrada.')

SparkSession cerrada.
